# Local setup and validation history
This notebook preserves Ebere’s VS Code setup, validation and API connection checks. Later API test cells are historical diagnostics; they are not required to reproduce the study. For the final judge use `analysis/blv_judge_run`; for human comparisons use `analysis/blv_current_evaluation`. Adapt image paths before running.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openpyxl
from PIL import Image, ImageOps

print("Python location:", sys.executable)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Notebook packages loaded.")

In [ ]:
# Find the project folder
current_folder = Path.cwd().resolve()

PROJECT_FOLDER = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / ".git").exists()
        and (folder / "sampling" / "outputs").is_dir()
    ),
    None
)

if PROJECT_FOLDER is None:
    raise FileNotFoundError(
        "Open the blv-project folder in VS Code, then try again."
    )

OUTPUT_FOLDER = PROJECT_FOLDER / "sampling" / "outputs"
FINAL_FOLDER = OUTPUT_FOLDER / "final_sample"
PILOT_FOLDER = OUTPUT_FOLDER / "blur_calibration"

# Read the existing sample and blur scores
sample_file = FINAL_FOLDER / "sample_manifest_with_blur.csv"
sample = pd.read_csv(sample_file)

# Check the saved records
expected_groups = {
    "blurry": 70,
    "clear": 70,
    "low_light": 60,
    "text": 50,
}

assert len(sample) == 250, "Expected 250 image records."
assert sample["file_name"].notna().all(), "Missing filenames."
assert sample["file_name"].is_unique, "Duplicate filenames found."
assert sample["laplacian_variance"].notna().all(), "Missing blur scores."
assert sample["stratum"].value_counts().to_dict() == expected_groups, (
    "The group counts do not match the agreed sample."
)

print("Project folder:", PROJECT_FOLDER)
print("Selected images:", len(sample))
print("Images with blur scores:", sample["laplacian_variance"].notna().sum())
print("\nGroup counts:")
print(sample["stratum"].value_counts())

In [ ]:
# Read the existing pilot scores and selected threshold
pilot = pd.read_csv(
    PILOT_FOLDER / "pilot_with_blur_scores.csv"
)

with open(PILOT_FOLDER / "selected_threshold.json") as file:
    threshold_settings = json.load(file)

# Check the pilot records
assert len(pilot) == 200, "Expected 200 pilot images."
assert pilot["file_name"].notna().all(), "Missing pilot filenames."
assert pilot["file_name"].is_unique, "Duplicate pilot images found."
assert pilot["laplacian_variance"].notna().all(), "Missing pilot blur scores."

# Check that the pilot and final sample contain different images
overlap = set(pilot["file_name"]) & set(sample["file_name"])
assert not overlap, f"Found {len(overlap)} overlapping images."

# Check the calibration and evaluation groups
expected_pilot_groups = {
    ("choose_threshold", "blurry"): 50,
    ("choose_threshold", "clear"): 50,
    ("check_threshold", "blurry"): 50,
    ("check_threshold", "clear"): 50,
}

actual_pilot_groups = (
    pilot.groupby(["purpose", "reference_group"]).size().to_dict()
)

assert actual_pilot_groups == expected_pilot_groups, (
    "The pilot groups do not match the expected split."
)

threshold = threshold_settings["threshold"]

print("Pilot images:", len(pilot))
print("Pilot images with blur scores:", pilot["laplacian_variance"].notna().sum())
print("Overlap with final sample:", len(overlap))
print("Saved threshold:", threshold)
print("\nPilot groups:")
print(pd.crosstab(pilot["purpose"], pilot["reference_group"]))

In [ ]:
# Current location of the image folders on your Mac
IMAGE_ROOT = Path.home() / "Desktop" / "blv-project" / "raw_data"

# Combine the final sample and pilot image lists
required_images = pd.concat(
    [
        sample[["file_name", "split"]],
        pilot[["file_name", "split"]],
    ],
    ignore_index=True,
)

image_problems = []

for _, row in required_images.iterrows():
    image_path = IMAGE_ROOT / row["split"] / row["file_name"]

    try:
        with Image.open(image_path) as image:
            image.load()
    except Exception as error:
        image_problems.append({
            "file_name": row["file_name"],
            "problem": str(error),
        })

print("Image location:", IMAGE_ROOT)
print("Images checked:", len(required_images))
print("Images successfully opened:", len(required_images) - len(image_problems))
print("Images needing attention:", len(image_problems))

if image_problems:
    display(pd.DataFrame(image_problems))

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass(
    "Paste your Gemini API key here: "
).strip()

print("Key loaded." if os.environ["GEMINI_API_KEY"] else "No key entered.")

In [ ]:
%pip install -q google-genai

In [ ]:
import os
from google import genai
from google.genai import types

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Run the private API-key entry cell first.")

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(timeout=30000),
)

available_models = list(client.models.list())

print("Connected to Gemini successfully.")
print("\nGemini models returned by the API:")

for model in available_models:
    if "gemini" in (model.name or "").lower():
        print(model.name)

In [ ]:
import os
from google import genai
from google.genai import types, errors

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(timeout=120000),
)

JUDGE_MODEL = "gemini-3.8-flash"

try:
    response = client.models.generate_content(
        model=JUDGE_MODEL,
        contents="Reply with exactly: Connection test successful.",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="low"),
            max_output_tokens=256,
        ),
    )

    print("Model version:", response.model_version)
    print("Response:", response.text)

except errors.APIError as error:
    print("API error code:", error.code)
    print("Message:", error.message)

In [ ]:
from google.genai import types, errors

TEST_MODEL = "gemini-3.1-flash-lite"

try:
    response = client.models.generate_content(
        model=TEST_MODEL,
        contents="Reply with just the word OK.",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="low"),
            max_output_tokens=256,
        ),
    )

    print("Model:", response.model_version)
    print("Response:", response.text)

except errors.APIError as error:
    print("API error code:", error.code)
    print("Message:", error.message)

In [ ]:
import os
from google import genai
from google.genai import types, errors

test_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(
        timeout=120000,
        retry_options=types.HttpRetryOptions(attempts=1),
    ),
)

print("Testing Gemini 3.8 Flash...", flush=True)

try:
    received_text = False

    for chunk in test_client.models.generate_content_stream(
        model="gemini-3.8-flash",
        contents="Reply with only the word OK.",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="low"),
            max_output_tokens=1024,
        ),
    ):
        if chunk.text:
            received_text = True
            print(chunk.text, end="", flush=True)

    print(
        "\nTest completed."
        if received_text
        else "\nThe request finished without a text response."
    )

except errors.APIError as error:
    print("\nAPI error code:", error.code)
    print("Message:", error.message)

finally:
    test_client.close()